Name: Sama Hany Soliman
ID: 30904041202581

In [ ]:
#importing libiraries
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt

## =>READING FILES
#connecting to database, extract tables from it and print them
conn = sqlite3.connect("/content/Level3_finial_project_library.db")
members_sql = pd.read_sql_query('SELECT * FROM members', conn)
books_sql = pd.read_sql_query('SELECT * FROM books', conn)
checkouts_sql = pd.read_sql_query('SELECT * FROM checkouts', conn)
print("Members table:\n", 'number of rows and columns:',members_sql.shape,'\n',members_sql.head())
print("===============================================")
print("\n Books table:\n", 'number of rows and columns:',books_sql.shape,'\n', books_sql.head())
print("===============================================")
print("\n Checkouts table:\n", 'number of rows and columns:',checkouts_sql.shape,'\n', checkouts_sql.head())
print("===============================================")
# reading json file and printing first 5 rows
book_catalog = pd.read_json("/content/Level3_finial_project_book_catelog.json")
print("\n Book catelog table:\n", 'number of rows and columns:',book_catalog.shape,'\n', book_catalog.head())
print("===============================================")
# reading html file, extrct table from it and print it
kickoff_html = pd.read_html("/content/Level3_finial_project_event_signups.html")
kickoff_table = kickoff_html[0]
print("\n Kickoff singup table:\n", 'number of rows and columns:',kickoff_table.shape,'\n',kickoff_table.head())

Explanation
I started by importing all the libraries I would use in the project, such as pandas, sqlite3, and others. Then, I connected to the database, read the tables inside it, and printed the first 5 rows from each table. After that, I read the JSON file and printed its first 5 rows. Finally, I read the HTML file and extracted the first table from it.

Answering Business questions about the library

In [ ]:
#1. How much is each member borrowing?
query1 = '''
SELECT members.member_id,
  members.first_name,
  members.last_name,
  COUNT (checkouts.checkout_id) AS member_count_checkout
FROM members
LEFT JOIN checkouts ON members.member_id = checkouts.member_id
GROUP BY members.member_id, members.first_name
'''
df_q1= pd.read_sql_query(query1 , conn)
print("Total count members' checkouts:", len(df_q1))
print(df_q1.head())

In [ ]:
#2.Which books match a chosen author pattern?
query2 = '''
SELECT book_id,
  title,
  author
FROM books
Where books.author LIKE 'S%'
'''
df_q2 = pd.read_sql_query(query2, conn)
print("Total matching books with pattern S:",len(df_q2))
print("\n",df_q2.head())

In [ ]:
#3.What are the most popular books?
query3 = '''
SELECT books.title,
  books.book_id,
  books.author,
  COUNT (checkouts.checkout_id) AS Total_checkout
From books
JOIN checkouts ON books.book_id = checkouts.book_id
GROUP BY books.title, books.book_id
ORDER BY Total_checkout DESC
LIMIT 5
'''
df_q3 = pd.read_sql_query(query3, conn)
print("Most Frequently Borrowed Books (Top 5):\n")
print(df_q3)

In [ ]:
#4.Who are the most active readers?
query4 = '''
SELECT members.member_id,
  members.first_name,
  members.last_name,
  COUNT (checkouts.checkout_id) AS Total_checkout
FROM members
JOIN checkouts ON checkouts.member_id = members.member_id
GROUP BY members.member_id, members.first_name
ORDER BY Total_checkout DESC
LIMIT 10
'''
df_q4= pd.read_sql_query(query4,conn)
print("Members with the Most Borrowed Books (Top 10):\n")
print(df_q4)

In [ ]:
#5.What does a neighborhood's activity look like further back in time?
query5 = '''
SELECT members.member_id,
  members.first_name,
  members.last_name,
  members.neighborhood,
  checkouts.checkout_date
FROM members
  JOIN checkouts ON members.member_id = checkouts.member_id
WHERE members.neighborhood = "Maadi"
ORDER BY checkouts.checkout_date DESC
LIMIT 10 OFFSET 10
'''
df_q5= pd.read_sql_query(query5,conn)
print("Second Set of 10 Checkouts for Maadi neighborhood:\n")
print(df_q5)

Files combination

In [ ]:
# first grouping members' counts
mem_count = checkouts_sql.groupby('member_id')['checkout_id'].count()
mem_count = mem_count.reset_index()
mem_count.columns = ['member_id', 'books_checkedout_count']
# Here, I assigned a variable, then took the checkout table and applied groupby to count each member's checkouts.
# Then, I transformed the data into a DataFrame to add it to the main database in a future step, and named the columns.
#==> add members' count to data then add members and checkouts and made them one table
mem_checkouts = pd.merge(members_sql , checkouts_sql, on='member_id', how='left')
print("Number of rows and columns of members and checkouts data combined:", mem_checkouts.shape,'\n')
print("Merging members with checkouts:\n \n",mem_checkouts.head())
print("\n===============================================\n")
mem_checkouts_count = pd.merge(mem_checkouts , mem_count , on='member_id', how='left')
print("Members checkouts counts number of rows and columns:", mem_checkouts_count.shape,'\n')
print("Merging members' checkouts counts with checkouts table:\n \n",mem_checkouts_count.head())
print("\n========================================\n\n")
#==> combing json file book catelog to dataframe
books = pd.merge(mem_checkouts_count , book_catalog, on= 'book_id', how= 'left')
print("Number of rows and columns after adding books details:", books.shape,'\n')
print("Data after adding book catalog:\n \n",books.head())
print("\n========================================\n\n")
#==> Combing kickoff signup html file with data and corect columns name to be same in dataframe
kickoff_table.columns = kickoff_table.columns.str.lower().str.strip().str.replace(' ', '_')
final_combined = pd.concat([books, kickoff_table], ignore_index=True, sort=False)
print("Number of rows and columns after adding kickoff signups:", final_combined.shape)
print("Number of rows in the final combination:" , final_combined.shape[0])
print("Number of columns in the final combination:" , final_combined.shape[1],'\n')
print("Data after adding kickoff signup:\n \n",final_combined.head())
print("\n========================================\n\n")
#Saving task1_combined_data file
final_combined.to_csv("EYOUTH-30904041202581_task1_combined_data.csv", index=False)
from google.colab import files
files.download("EYOUTH-30904041202581_task1_combined_data.csv")
print("EYOUTH-30904041202581_task1_combined_data.csv is saved")

Explanation
I started by merging the files together. First, I got each member's checkout count using groupby(). Then, I reset the index to convert the result into a DataFrame and saved it in a variable. After that, I merged the members table with the checkouts table using the common column, which is member_id. Then, I added the member checkout count by merging it with the combined data. After that, I merged the book catalog JSON file using the book_id column. Next, I standardized the HTML file format and corrected its column names before concatenating it with the data. Finally, I saved the combined dataset as a CSV file

Task2: Data integrity

In [ ]:
# Reading csv file and show some details about it
df = pd.read_csv("EYOUTH-30904041202581_task1_combined_data.csv")
print("Data columns name:",df.columns.to_list())
print("\n----------------------------------------------")
print("\nData info:\n",df.info())
print("\n========================================\n\n")
#==>Detecting Duplicates and removing them
print("Number of duplicated rows:", df.duplicated().sum())
print("========================================")
print("\nDuplicated Data:\n",df[df.duplicated(keep=False)])
df = df.drop_duplicates()
print("========================================")
print("Number of duplicated rows after removing:", df.duplicated().sum())
print("\nNumber of rows and columns after removing duplicates:", df.shape)
print("\n========================================\n\n")

#==>Detecting Missing Values
print("Number of missing values:\n",df.isna().sum())
print("\n========================================\n\n")
member_columns = ['first_name','last_name', 'grade','neighborhood','membership_status','join_date']
for column in member_columns:
    df[column] = df.groupby('member_id')[column].transform('first')

book_columns = [ 'genre','pages','publication_year','publisher']
for column in book_columns:
    df[column] = df.groupby('book_id')[column].transform('first')
print("Number of missing values after filling data that related to member_id & book_id:\n",df.isna().sum())
print("\n========================================\n\n")
#Problem: Checkouts with no matching member
  # here i get the members that there is no data about them and drop them
print("\nseeing number of member_id that doesn't exist:",df['first_name'].isna().sum())
print("\nseeing member_id that doesn't exist:\n",df[df['first_name'].isna()])
print("\n========================================\n\n")
df= df.dropna(subset=['first_name'])
print("\nseesing data after dropping members that don't exist:\n",df[df['first_name'].isna()])
print("Number of missing values:\n",df.isna().sum())
print("\n========================================\n")

#Filling grade column:
print("Number of missing values in grade column before cleaning:",df['grade'].isna().sum())
df['grade'] = df['grade'].fillna(df['grade'].median()).astype(int)
print("Number of missing values in grade column after cleaning:",df['grade'].isna().sum())
print("\n----------------------------------------------")
#Filling join date column:
print("Number of missing values in join_date column before cleaning:",df['join_date'].isna().sum())
df['join_date'] = df['join_date'].fillna(df['join_date'].mode()[0])
print("Number of missing values in join_date column after cleaning:",df['join_date'].isna().sum())
print("\n----------------------------------------------")
#Filling ckeckout id column:
print("Number of missing values in checkout_id column before cleaning:",df['checkout_id'].isna().sum())
df['checkout_id'] = df['checkout_id'].fillna(df['checkout_id'].median()).astype(int)
print("Number of missing values in checkout_id column after cleaning:",df['checkout_id'].isna().sum())
print("\n----------------------------------------------")
#Filling book id column:
print("Number of missing values in book_id column before cleaning:",df['book_id'].isna().sum())
df['book_id'] = df['book_id'].fillna(df['book_id'].median()).astype(int)
print("Number of missing values in book_id column after cleaning:",df['book_id'].isna().sum())
print("\n----------------------------------------------")
#Filling checkout date column:
print("Number of missing values in checkout_date column before cleaning:",df['checkout_date'].isna().sum())
df['checkout_date'] = df['checkout_date'].fillna(df['checkout_date'].mode()[0])
print("Number of missing values in checkout_date column after cleaning:",df['checkout_date'].isna().sum())
print("\n----------------------------------------------")
#Filling return date column:
print("Number of missing values in return_date column before cleaning:",df['return_date'].isna().sum())
df['return_date'] = df['return_date'].fillna('Not Returned')
print("Number of missing values in return_date column after cleaning:",df['return_date'].isna().sum())
print("\n----------------------------------------------")
#Filling books checkedout count date column:
print("Number of missing values in books_checkedout_count column before cleaning:",df['books_checkedout_count'].isna().sum())
df['books_checkedout_count'] = df['books_checkedout_count'].fillna(1).astype(int)
print("Number of missing values in books_checkedout_count column after cleaning:",df['books_checkedout_count'].isna().sum())
print("\n----------------------------------------------")
#Filling genre column:
print("Number of missing values in genre column before cleaning:",df['genre'].isna().sum())
df['genre'] = df['genre'].fillna(df['genre'].mode()[0])
print("Number of missing values in genre column after cleaning:",df['genre'].isna().sum())
print("\n----------------------------------------------")
#Filling pages column:
print("Number of missing values in pages column before cleaning:",df['pages'].isna().sum())
df['pages'] = df['pages'].fillna(df['pages'].median()).astype(int)
print("Number of missing values in pages column after cleaning:",df['pages'].isna().sum())
print("\n----------------------------------------------")
#Filling publication year column:
print("Number of missing values in publication_year column before cleaning:",df['publication_year'].isna().sum())
df['publication_year'] = df['publication_year'].fillna(df['publication_year'].median()).astype(int)
print("Number of missing values in publication_year column after cleaning:",df['publication_year'].isna().sum())
print("\n----------------------------------------------")
#Filling publisher column:
print("Number of missing values in publisher column before cleaning:",df['publisher'].isna().sum())
df['publisher'] = df['publisher'].fillna(df['publisher'].mode()[0])
print("Number of missing values in publisher column after cleaning:",df['publisher'].isna().sum())


print("\n========================================\n\n")
print("Number of missing values:\n",df.isna().sum())
print("\n========================================\n\n")
#==>Inconsistent form
# I checked form of every object column the showed only columns that have inconsistent form
# inconsistent form of neighborhood column
print(df['neighborhood'].value_counts())
print("\n----------------------------------------------")
df['neighborhood'] = df['neighborhood'].str.strip().str.title()
df['neighborhood'] = df['neighborhood'].replace({'Nasr  City':'Nasr City'})
print(df['neighborhood'].value_counts())
print("\n----------------------------------------------\n")
# inconsistent form of membership status column
print(df['membership_status'].value_counts())
print("\n----------------------------------------------")
df['membership_status'] = df['membership_status'].str.strip().str.title()
print(df['membership_status'].value_counts())
print("\n----------------------------------------------")
# inconsistent form of publisher column
print(df['publisher'].value_counts())
print("\n----------------------------------------------")
df['publisher'] = df['publisher'].str.strip().str.title()
print(df['publisher'].value_counts())
print("\n----------------------------------------------")
# inconsistent form of genre column
print(df['genre'].value_counts())
print("\n----------------------------------------------")
df['genre'] = df['genre'].str.strip().str.title()
print(df['genre'].value_counts())
print("===============================================================\n")

print(df.head())
print("\nNumber of rows and columns:",df.shape)
print("===============================================================\n")
#Saving task2_Data_Integrity file
df.to_csv("EYOUTH-30904041202581_task2_cleaned_data.csv", index=False)
from google.colab import files
files.download("EYOUTH-30904041202581_task2_cleaned_data.csv")
print("EYOUTH-30904041202581_task2_cleaned_data.csv is saved")

Explanation
I read the combined dataset and checked the number of rows and columns. I used info() to view the data type of each column and the number of non-null values. Then, I checked for duplicate records, displayed them, removed them, and confirmed that no duplicates remained. After that, I examined the number of missing values. I noticed that some member_id values were repeated, so I used them to fill missing information for the same member, and I applied the same approach to book_id to fill missing book information. I also found some member_id values with no recorded information, so I removed those records because they represented only a small portion of the dataset and there was no reliable way to fill their missing values. Next, I filled the remaining missing values column by column. I used the median for the grade, checkout_id, book_id, pages, and publication_year columns because they are numerical columns and the median is less affected by outliers. I used the mode for the join_date, checkout_date, genre, and publisher columns because it represents the most frequent value and is suitable for these columns. In the return_date column, I replaced missing values with "Not Returned", and in the books_checkedout_count column, I filled missing values with 1, assuming the book had been checked out once. I also noticed that the neighborhood, membership_status, publisher, and genre columns had inconsistent formatting, such as extra spaces or different capitalization styles, so I cleaned them by removing unnecessary spaces and converting the text to title case. Finally, I saved the cleaned dataset as a CSV file.

Task3: Data Fairness and Version Control

In [ ]:
members_by_neighborhood = (df.groupby('neighborhood')['member_id'].nunique().reset_index(name='members_count'))

checkouts_by_neighborhood = (df.groupby('neighborhood')['checkout_id'].count().reset_index(name='checkouts_count'))

comparison = pd.merge(
    members_by_neighborhood,
    checkouts_by_neighborhood,
    on='neighborhood'
)

print(comparison)
print("\n==============================================\n")

comparison.plot(
    x='neighborhood',
    y=['members_count', 'checkouts_count'],
    kind='bar'
)

plt.title('Members and Checkouts by Neighborhood')
plt.xlabel('Neighborhood')
plt.ylabel('Count')
plt.xticks(rotation=45)
plt.show()